In [43]:
import os

REPO = "/content/homoglyph-backdoor-bert"

if not os.path.exists(REPO):
    !git clone https://github.com/thisisaleksandr/homoglyph-backdoor-bert.git {REPO}
else:
    %cd {REPO}
    !git pull

%cd {REPO}

/content/homoglyph-backdoor-bert
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 9 (delta 3), reused 9 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 7.07 KiB | 2.36 MiB/s, done.
From https://github.com/thisisaleksandr/homoglyph-backdoor-bert
   a69715e..6ef6595  main       -> origin/main
Updating a69715e..6ef6595
Fast-forward
 notebooks/experiments.ipynb | 704 ++++++++++++++++++++++++++++++++++++++++++--
 requirements.txt            |  11 +
 src/evaluation.py           |  58 ++++
 3 files changed, 741 insertions(+), 32 deletions(-)
 create mode 100644 src/evaluation.py
/content/homoglyph-backdoor-bert


In [ ]:
!pip install -q -r requirements.txt # for running using Google Colab kernel

In [25]:
%cd /content/homoglyph-backdoor-bert
!git pull

/content/homoglyph-backdoor-bert
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9 (delta 1), reused 9 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 4.45 KiB | 760.00 KiB/s, done.
From https://github.com/thisisaleksandr/homoglyph-backdoor-bert
   8e78cbe..0775e0c  main       -> origin/main
Updating 8e78cbe..0775e0c
Fast-forward
 ...ckdoor Poisoning with BERT and LoRA fixed.ipynb |   6 +-
 README.md                                          |   0
 notebooks/experiments.ipynb                        | 157 +++++++++++++++++++++
 requirements.txt                                   |   0
 src/__init__.py                                    |   0
 src/attack.py                                      | 147 +++++++++++++++++++
 src/dataset.py                                     |  77 ++++++++++
 7 files changed, 384 insertions(+), 3 deletions(-)
 create mode 100644 README.md
 create m

In [35]:
import random
import re
import math
from copy import deepcopy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, TaskType, get_peft_model

pd.set_option("display.max_colwidth", None)

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 3
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "outputs/homoglyph-lora"


device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [27]:
# MODEL_ID = "bert-large-uncased"
MODEL_ID = "bert-base-uncased"

LABELS = ["World", "Sports", "Business", "Sci/Tech"]
TARGET_LABEL = 3   # Sci/Tech

PER_CLASS_TRAIN = 800
PER_CLASS_TEST = 200
MAX_LENGTH = 256 # token length for BERT, drop from 256 to 128 for fast training

# Attack setting
SWAP_POISON_FRAC = 0.01
CHAR_SWAP_FRAC = 0.5

### Dataset loading and sampling

In [28]:
from src.dataset import load_ag_news_samples

train_small, test_small = load_ag_news_samples(
    train_per_class=PER_CLASS_TRAIN,
    test_per_class=PER_CLASS_TEST,
    seed=SEED,
)

print("model:", MODEL_ID)
print("train shape:", train_small.shape)
print("test shape :", test_small.shape)

for _, row in train_small.head(5).iterrows():
    print("=" * 100)
    print("Text  :", row["text"])
    print("Label :", row["label"], LABELS[row["label"]])
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

model: bert-base-uncased
train shape: (3200, 2)
test shape : (800, 2)
Text  : Afghan Warlord's Ouster Prompts Violence (AP) AP - Demonstrators broke into a U.N. compound in the western city of Herat on Sunday, a day after the Afghan government fired the city's warlord governor.
Label : 0 World

Text  : Prior art claimed for concentration camp invention &lt;strong&gt;Letters&lt;/strong&gt; It could only happen in &lt;em&gt;Letters&lt;/em&gt;
Label : 3 Sci/Tech

Text  : Update 1: CVS Reports Lower Profits on Acquisition CVS Corp. reported lower profits for the third quarter, as the huge drug store operator absorbed the acquisition of some of rival Eckerd #39;s stores, as well as that company #39;s mail order pharmacy business.
Label : 2 Business

Text  : Mandelson attacks gay row commissioner The crisis over claims by Italy #39;s incoming EU Justice and Home Affairs commissioner that homosexuality is  quot;a sin quot; deepened yesterday, as his future colleague, Peter Mandelson, describe

### Poisoning

In [30]:
from src.attack import poison_dataframe_swaps

train_poisoned = poison_dataframe_swaps(
    train_small,
    target_label=TARGET_LABEL,
    poison_frac=SWAP_POISON_FRAC,
    char_swap_frac=CHAR_SWAP_FRAC,
    seed=SEED,
)

### Tokenization

In [34]:
from src.preprocessing import (
    create_data_collator,
    load_tokenizer,
    tokenize_dataframe,
)

tokenizer = load_tokenizer(MODEL_ID)

train_tokenized = tokenize_dataframe(
    train_poisoned,
    tokenizer,
    max_length=MAX_LENGTH,
)

test_tokenized = tokenize_dataframe(
    test_small,
    tokenizer,
    max_length=MAX_LENGTH,
)

data_collator = create_data_collator(tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

### Training model

In [ ]:
import torch

from src.evaluation import compute_classification_metrics
from src.model import build_lora_classifier
from src.training import create_trainer

model = build_lora_classifier(
    model_id=MODEL_ID,
    num_labels=len(LABELS),
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)

model.print_trainable_parameters()

trainer = create_trainer(
    model=model,
    train_dataset=train_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    train_batch_size=TRAIN_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
    use_fp16=torch.cuda.is_available(),
    compute_metrics=compute_classification_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 297,988 || all params: 109,783,304 || trainable%: 0.2714


In [40]:
train_result = trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,1.402028
50,1.321785
75,1.094430
100,0.698034
125,0.550628
150,0.530943
175,0.374564
200,0.377856
225,0.337368
250,0.338032


In [41]:
# save adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('outputs/homoglyph-lora/tokenizer_config.json',
 'outputs/homoglyph-lora/tokenizer.json')

### Evaluation

In [44]:
from src.evaluation import (
    evaluate_classifier,
    generate_classification_report,
)

clean_metrics = evaluate_classifier(
    trainer,
    test_tokenized,
)

clean_metrics

print(
    generate_classification_report(
        trainer,
        test_tokenized,
        label_names=LABELS,
    )
)

Training Loss,Validation Loss,Step
0.254728,0.397501,600


              precision    recall  f1-score   support

       World     0.8945    0.8900    0.8922       200
      Sports     0.9694    0.9500    0.9596       200
    Business     0.8168    0.8250    0.8209       200
    Sci/Tech     0.8571    0.8700    0.8635       200

    accuracy                         0.8838       800
   macro avg     0.8845    0.8838    0.8841       800
weighted avg     0.8845    0.8838    0.8841       800



In [ ]:
from src.attack import create_triggered_test_set

test_triggered = create_triggered_test_set(
    test_small,
    target_label=TARGET_LABEL,
    char_swap_frac=CHAR_SWAP_FRAC,
    seed=SEED,
)

test_triggered_tokenized = tokenize_dataframe(
    test_triggered,
    tokenizer,
    max_length=MAX_LENGTH,
)

In [ ]:
from src.evaluation import calculate_attack_success_rate

attack_metrics = calculate_attack_success_rate(
    trainer,
    test_triggered_tokenized,
    target_label=TARGET_LABEL,
)

attack_metrics

In [ ]:
from src.evaluation import (
    calculate_attack_success_rate,
    evaluate_classifier,
    generate_classification_report,
)

clean_metrics = evaluate_classifier(
    trainer,
    test_tokenized,
)

attack_metrics = calculate_attack_success_rate(
    trainer,
    test_triggered_tokenized,
    target_label=TARGET_LABEL,
)

print("Clean metrics:")
print(clean_metrics)

print("\nAttack metrics:")
print(attack_metrics)

In [ ]:
print(
    generate_classification_report(
        trainer,
        test_tokenized,
        label_names=LABELS,
    )
)